<a href="https://colab.research.google.com/github/KDY0829/TD_Learning/blob/main/TD_Learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 과제
- 랜덤 벽 GridWorld에서 TD Learning 구현
- 랜덤하게 생성되는 벽이 존재하는 GridWorld 환경을 만들고, 에이전트가 랜덤 정책으로 움직일 때 TD Learning을 이용해 상태가치 V(s)를 학습하는 프로그램을 작성

- 4*4 GridWorld 환경 구현
- 시작 위치 (0, 0)
- 목표 위치 (3, 3)
- 행동 -> 0: 왼쪽, 1: 위, 2: 오른쪽, 3: 아래
- 매 에피소드가 시작될 때마다 격자 안에 벽을 랜덤하게 생성(이동 불가)
    - 벽의 개수는 2개 또는 3개
    - 시작위치에 벽이 생성되면 안 됨
    - 목표 위치에 벽이 생성되면 안 됨
    - 같은 위치에 벽이 중복 생성되면 안 됨
- 이동하려는 위치가 빈 칸이면 이동 가능
    - 격자를 벗어나면 원래 위치에 그대로 둠
    - 벽이 있는 칸으로 이동하면 제자리 유지
- 보상: 일반 이동 -1
- 목표: (3, 3)에 도착하면 에피소드 종료

> 일반 그리드 환경과 벽 그리드 환경과의 상태가치 차이에 대해 작성

In [1]:
import random
import numpy as np
from tqdm import tqdm

In [2]:
class RandomWallGridWorld():
    def __init__(self):
        self.size = 4
        self.start = (0, 0)
        self.goal = (3, 3)
        self.walls = []
        self.reset()

    def reset(self):
        self.x, self.y = self.start
        candidates = [(r, c) for r in range(self.size) for c in range(self.size) if (r, c) != self.goal]
        num_walls = random.randint(2, 3)
        self.walls = random.sample(candidates, num_walls)
        return (self.x, self.y)

    def step(self, a):
        # 0:왼쪽, 1:위, 2:오른쪽, 3:아래
        prev_x, prev_y = self.x, self.y

        if a == 0: self.y -= 1
        elif a == 1: self.x -= 1
        elif a == 2: self.y += 1
        elif a == 3: self.x += 1

        # 격자 밖으로 나가거나 벽을 만난 경우 위치 복구 (제자리 유지)
        if (self.x < 0 or self.x >= self.size or
            self.y < 0 or self.y >= self.size or
            (self.x, self.y) in self.walls):
            self.x, self.y = prev_x, prev_y

        reward = -1 # 이동 시 보상은 항상 -1
        done = self.is_done()
        return (self.x, self.y), reward, done

    def is_done(self):
        return (self.x, self.y) == self.goal

    def get_state(self):
        return (self.x, self.y)


In [3]:
class Agent():
    def select_action(self):
        return random.randint(0, 3)

In [4]:
def main():
    env = RandomWallGridWorld()
    agent = Agent()
    data = [[0.0 for _ in range(4)] for _ in range(4)]

    gamma = 1.0
    alpha = 0.01
    episodes = 10000  # 1. 에피소드 횟수 현실적으로 조정
    max_steps = 500   # 2. 한 에피소드당 최대 걸음 수 제한 (무한 루프 방지)

    # 진행 상황을 보기 위해 tqdm 사용
    for k in tqdm(range(episodes), desc="학습 중"):
        state = env.reset()
        done = False
        steps = 0

        while not done and steps < max_steps:
            curr_x, curr_y = state
            action = agent.select_action()
            (next_x, next_y), reward, done = env.step(action)

            if done:
                td_target = reward
            else:
                td_target = reward + gamma * data[next_x][next_y]

            data[curr_x][curr_y] += alpha * (td_target - data[curr_x][curr_y])
            state = (next_x, next_y)
            steps += 1

    print("\n--- 학습 완료 ---")
    for row in data:
        print([round(v, 2) for v in row])

In [5]:
if __name__ == '__main__':
    main()

학습 중: 100%|██████████| 10000/10000 [00:01<00:00, 6356.70it/s]


--- 학습 완료 ---
[-102.66, -99.2, -93.04, -89.85]
[-100.75, -94.79, -87.05, -80.93]
[-94.41, -88.56, -73.66, -45.78]
[-89.52, -83.17, -52.57, 0.0]
